# ⚖️ Página 2 del BI — Comparación A vs B

Replica la segunda página del BI oficial: selectores jerárquicos (GITT / PRB / Regional), dos gráficas de barras espejadas y los labels analíticos finales (Hallazgo Principal, Resumen Desempeño, Estados y Brecha).

Usamos el mismo helper `loaders.py` de Página 1 para cargar desde Excel o DB indistintamente.

## 1. Imports y carga de datos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "src"))
from loaders import load_from_excel, load_from_db, describe_dict, compare_sources

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

In [ ]:
# Excel
data_excel = load_from_excel(Path("data/Metas 2026_GITT_ Para power BI.xlsx"))

# DB (API)
try:
    data_db = load_from_db("http://localhost/api", "admin", "Admin@UBPD2024!")
    print("✓ Ambas fuentes cargadas")
except Exception as exc:
    print(f"⚠️  Solo Excel disponible: {exc}")
    data_db = None

## 2. Descripción y comparación rápida

In [ ]:
if data_db is not None:
    compare_sources(data_excel, data_db)

## 3. Elegir fuente activa

Con la misma convención que el Notebook 1 — cambia `SOURCE` para alternar.

In [ ]:
SOURCE = "excel"   # "excel" | "db"

data = data_excel if SOURCE == "excel" else data_db
print(f"🔵 Fuente activa: {SOURCE}")

## 4. Preparar tablas

Limpieza idéntica a la Página 1 + merge con `PRB` para tener `Regional` y `GITT` en cada fila.

In [ ]:
hist = data["Historico"].copy()
prb  = data["PRB"].copy()

# Limpieza
hist = hist.dropna(subset=["CodPRB", "Cod_Indicador", "AÑO"])
hist = hist[(hist["CodPRB"] != 0) & (hist["Cod_Indicador"] != 0)]
for col in ["Meta", "Avance Total", "Línea Base 2025"]:
    if col in hist.columns:
        hist[col] = pd.to_numeric(hist[col], errors="coerce").fillna(0)

# Merge para tener Regional/GITT/PRB_nombre
enriched = hist.merge(
    prb[["COD", "PRB", "Regional", "GITT"]].rename(columns={"PRB": "PRB_nombre"}),
    left_on="CodPRB", right_on="COD", how="left",
)
enriched.head(3)

## 5. Selectores jerárquicos disponibles

El BI ofrece un dropdown en árbol con 3 niveles:
- **GITT** → opciones: `ANTIOQUIA`, `ARAUCA`, `ATLÁNTICO`…
- **PRB**  → opciones: `Alta Y Media Guajira`, `Alto Putumayo`…
- **Regional** → opciones: `CENTRO`, `NOROCCIDENTE`, `NORORIENTE`…

In [ ]:
niveles = {
    "gitt":     sorted(prb["GITT"].dropna().unique().tolist()),
    "prb":      sorted(prb["PRB"].dropna().unique().tolist()),
    "regional": sorted(prb["Regional"].dropna().unique().tolist()),
}

for nivel, opciones in niveles.items():
    print(f"{nivel.upper():10} → {len(opciones)} opciones. Primeras 5: {opciones[:5]}")

## 6. Función de fetch por grupo

Dado un `(nivel, valor)` retorna por cada indicador su `Meta` y `Avance`. Es la misma lógica que implementa el backend en `/api/bi/comparison`.

In [ ]:
def fetch_grupo(df: pd.DataFrame, nivel: str, valor: str | None, anio: int = 2026) -> pd.DataFrame:
    """
    Filtra el DataFrame enriched por (nivel, valor) y agrupa por indicador.
    Devuelve columnas: codigo, nombre_largo, meta, avance, base_2025, pct.
    El pct (= avance/meta) es lo que el BI dibuja en las barras.
    """
    sub = df[df["AÑO"] == anio]
    if   nivel == "gitt"     and valor: sub = sub[sub["GITT"]       == valor]
    elif nivel == "regional" and valor: sub = sub[sub["Regional"]   == valor]
    elif nivel == "prb"      and valor: sub = sub[sub["PRB_nombre"] == valor]

    agg = (
        sub.groupby(["Código del Indicador", "Indicador"], as_index=False)
           .agg(
               meta      = ("Meta",            "sum"),
               avance    = ("Avance Total",    "sum"),
               base_2025 = ("Línea Base 2025", "sum"),
           )
    )
    agg["pct"] = np.where(agg["meta"] > 0, agg["avance"] / agg["meta"], 0.0)
    return agg.rename(columns={"Indicador": "nombre_largo"}).reset_index(drop=True)


## 7. Selección A y B

Cambia `SEL_A` y `SEL_B` para comparar distintos pares. Por ejemplo:
- `("gitt", "ANTIOQUIA")` vs `("gitt", "META")` (global)
- `("regional", "CENTRO")` vs `("regional", "NOROCCIDENTE")`

In [ ]:
# (nivel, valor)  — valor=None si nivel=="meta"
SEL_A = ("gitt",     "ANTIOQUIA")
SEL_B = ("regional", "NOROCCIDENTE")

grupo_a = fetch_grupo(enriched, SEL_A[0], SEL_A[1])
grupo_b = fetch_grupo(enriched, SEL_B[0], SEL_B[1])

print(f"Grupo A: {SEL_A}  →  {len(grupo_a)} indicadores")
print(f"Grupo B: {SEL_B}  →  {len(grupo_b)} indicadores")

In [ ]:
grupo_a

In [ ]:
grupo_b

## 8. Unificar en un solo DataFrame para graficar

Por cada indicador necesitamos `a` (avance del grupo A) y `b` (avance del grupo B).

In [ ]:
# Por cada indicador necesitamos pct_a y pct_b (= avance/meta)
merged = pd.merge(
    grupo_a[["Código del Indicador", "nombre_largo", "avance", "meta", "base_2025", "pct"]]
        .rename(columns={"avance": "avance_a", "meta": "meta_a", "base_2025": "base_2025_a", "pct": "pct_a"}),
    grupo_b[["Código del Indicador", "avance", "meta", "base_2025", "pct"]]
        .rename(columns={"avance": "avance_b", "meta": "meta_b", "base_2025": "base_2025_b", "pct": "pct_b"}),
    on="Código del Indicador", how="outer",
).fillna(0)
merged


## 9. Dos gráficas de barras espejadas

Igual que el BI: A a la izquierda (crece hacia la izquierda), B a la derecha.

In [ ]:
# Etiquetas de Página 2 (DIFERENTES de Página 1) y orden propio
BI_PAGE2_ORDER = [
    "L1R-006-007", "L1A-021", "L1A-022", "L1R-001", "L1R-004",
    "L1P-002", "L1A-020a", "L1P-006", "L1P-010", "L1R-003",
    "L1P-008-009", "L1R-005", "L1R-008",
]
BI_PAGE2_LABELS = {
    "L1R-006-007": "SIF (Confirmados y descartados)",
    "L1A-021":     "SB Mejoradas",
    "L1A-022":     "Postulados BI para verificación",
    "L1R-001":     "Encontradas con vida asignadas",
    "L1R-004":     "Personas con contacto exitoso",
    "L1P-002":     "PDD con solicitud de Búsqueda",
    "L1A-020a":    "PDD con Muestra asociada",
    "L1P-006":     "Planes Formulados con Aportantes",
    "L1P-010":     "Lugares Caracterizados",
    "L1R-003":     "Informes acaecidos finalizados",
    "L1P-008-009": "Informes Investigación con Hipótesis",
    "L1R-005":     "Entregas Dignas Asignadas",
    "L1R-008":     "Cuerpos Recuperados",
}

# Reordenar merged según el orden BI Página 2
_order = {c: i for i, c in enumerate(BI_PAGE2_ORDER)}
merged["__order"] = merged["Código del Indicador"].map(_order).fillna(9999)
merged = merged.sort_values("__order").drop(columns="__order").reset_index(drop=True)

labels_plot = [BI_PAGE2_LABELS.get(c, c) for c in merged["Código del Indicador"]]

# IMPORTANTE: las barras del BI muestran pct = avance/meta (fracción 0..1)
maxabs = max(merged["pct_a"].max(), merged["pct_b"].max(), 0.05)
y = np.arange(len(merged))

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 7), sharey=True,
                                  gridspec_kw={"wspace": 0.35})

# Grupo A — eje X invertido
ax_a.barh(y, merged["pct_a"], color="#9B5F7E", height=0.5)
ax_a.set_xlim(maxabs, 0)
ax_a.invert_yaxis()  # primer indicador (SIF) ARRIBA, igual que el BI
ax_a.set_yticks(y)
ax_a.set_yticklabels(labels_plot)
ax_a.yaxis.tick_right()
ax_a.set_title(f"Grupo A — {SEL_A[0].upper()} | {SEL_A[1]}", color="#7C3A5C", fontweight="bold")
ax_a.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(merged["pct_a"]):
    if v > 0:
        ax_a.text(v, i, f" {v:.2f}", va="center", ha="left", fontsize=9, color="#555")

# Grupo B
ax_b.barh(y, merged["pct_b"], color="#35678A", height=0.5)
ax_b.set_xlim(0, maxabs)
ax_b.invert_yaxis()
ax_b.set_title(f"Grupo B — {SEL_B[0].upper()} | {SEL_B[1]}", color="#35678A", fontweight="bold")
ax_b.spines[["top", "right", "left"]].set_visible(False)
ax_b.tick_params(left=False, labelleft=False)
for i, v in enumerate(merged["pct_b"]):
    if v > 0:
        ax_b.text(v, i, f" {v:.2f}", va="center", ha="left", fontsize=9, color="#555")

plt.suptitle("Comparación A vs B — Página 2 del BI (% = avance/meta)",
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()


## 10. Labels analíticos de la parte inferior

En el BI aparecen 5 cuadros:

| # | Cuadro | Fórmula |
|---|---|---|
| 1 | Hallazgo Principal | Indicador con mayor gap absoluto entre A y B, con la ventaja expresada como % |
| 2 | Resumen Desempeño | Comparación narrativa del % de ejecución de cada grupo |
| 3 | Estado Selección Filtro A | 🟢/🟡/🟠/🔴 según % de ejecución |
| 4 | Estado Selección Filtro B | idem |
| 5 | Brecha A vs B | Diferencia absoluta con indicación de quién supera |

In [ ]:
# Vista rápida de las nuevas columnas pct_a / pct_b
merged[["Código del Indicador", "nombre_largo", "avance_a", "meta_a", "pct_a",
                                                "avance_b", "meta_b", "pct_b"]]


In [ ]:
# ── Cálculo de KPIs globales (pct = avance/meta) ─────────────────────────
total_avance_a, total_meta_a = merged["avance_a"].sum(), merged["meta_a"].sum()
total_avance_b, total_meta_b = merged["avance_b"].sum(), merged["meta_b"].sum()
pct_a = total_avance_a / total_meta_a * 100 if total_meta_a else 0
pct_b = total_avance_b / total_meta_b * 100 if total_meta_b else 0

label_a = SEL_A[1] if SEL_A[1] else SEL_A[0].upper()
label_b = SEL_B[1] if SEL_B[1] else SEL_B[0].upper()
grupo_txt_a = f"{SEL_A[0].upper()} {label_a}"
grupo_txt_b = f"{SEL_B[0].upper()} {label_b}"

# ── BANNER ───────────────────────────────────────────────────────────────
diff_global = pct_a - pct_b
if diff_global > 0:
    banner = f"Comparando {grupo_txt_a} con {grupo_txt_b}, {label_a} presenta un mejor nivel de avance 2026, con una diferencia de {abs(diff_global):.1f}%."
elif diff_global < 0:
    banner = f"Comparando {grupo_txt_a} con {grupo_txt_b}, {label_b} presenta un mejor nivel de avance 2026, con una diferencia de {abs(diff_global):.1f}%."
else:
    banner = f"Comparando {grupo_txt_a} con {grupo_txt_b}, ambos presentan el mismo nivel de avance 2026."

# ── HALLAZGO PRINCIPAL ───────────────────────────────────────────────────
merged["gap_pct"] = (merged["pct_a"] - merged["pct_b"]) * 100
top = merged.loc[merged["gap_pct"].abs().idxmax()]
ventaja_label = label_a if top["gap_pct"] > 0 else label_b
otro_label    = label_b if top["gap_pct"] > 0 else label_a
hallazgo = (
    f"La mayor ventaja se observa en el indicador '{top['nombre_largo']}', "
    f"donde {ventaja_label} supera a {otro_label} en {abs(top['gap_pct']):.1f}%."
)

# ── RESUMEN DESEMPEÑO (mejor/peor por grupo, tie-break por orden BI) ─────
def mejor_peor(df, pct_col):
    if df.empty:
        return None, None
    val = df.copy()
    val["__o"] = val["Código del Indicador"].map(_order).fillna(9999)
    mejor = val.sort_values([pct_col, "__o"], ascending=[False, True]).iloc[0]["nombre_largo"]
    peor  = val.sort_values([pct_col, "__o"], ascending=[True,  True]).iloc[0]["nombre_largo"]
    return mejor, peor

mejor_a, peor_a = mejor_peor(merged, "pct_a")
mejor_b, peor_b = mejor_peor(merged, "pct_b")
resumen = (
    f"{label_a}: mejor desempeño en '{mejor_a}', mayor rezago en '{peor_a}'. "
    f"{label_b}: mejor desempeño en '{mejor_b}', mayor rezago en '{peor_b}'."
)

# ── ESTADO ───────────────────────────────────────────────────────────────
def tier(pct: float, avance: float):
    if avance <= 0:                    return "SIN DATO",   "Sin dato"
    if pct >= 70:                      return "ÓPTIMO",     "🟢 Óptimo"
    if pct >= 40:                      return "MODERADO",   "🟡 Moderado"
    if pct >= 10:                      return "EN PROGRESO","🟠 En progreso"
    return                                "CRÍTICO",        "🔴 Crítico"

ka, lblA = tier(pct_a, total_avance_a)
kb, lblB = tier(pct_b, total_avance_b)
estado_a = f"{label_a} | {lblA}"
estado_b = f"{label_b} | {lblB}"
estado_a_explic = (f"{label_a}: sin información disponible." if ka == "SIN DATO" else
                   f"{label_a} se clasifica como {ka} ({pct_a:.1f}%), explicado principalmente por rezagos en '{peor_a}' y bajos niveles de ejecución general.")
estado_b_explic = (f"{label_b}: sin información disponible." if kb == "SIN DATO" else
                   f"{label_b} se clasifica como {kb} ({pct_b:.1f}%), explicado principalmente por rezagos en '{peor_b}' y bajos niveles de ejecución general.")

# ── BRECHA ───────────────────────────────────────────────────────────────
if   diff_global > 0:  brecha = f"A supera a B en {abs(diff_global):.1f}%"
elif diff_global < 0:  brecha = f"B supera a A en {abs(diff_global):.1f}%"
else:                  brecha = "A y B presentan el mismo % de avance."

print("─" * 70)
print("BANNER");                            print(f"  {banner}")
print("─" * 70)
print("HALLAZGO PRINCIPAL");                print(f"  {hallazgo}")
print("─" * 70)
print("RESUMEN DESEMPEÑO");                 print(f"  {resumen}")
print("─" * 70)
print("ESTADO SELECCIÓN FILTRO A");         print(f"  {estado_a}");  print(f"  {estado_a_explic}")
print("ESTADO SELECCIÓN FILTRO B");         print(f"  {estado_b}");  print(f"  {estado_b_explic}")
print("─" * 70)
print("BRECHA A vs B");                     print(f"  {brecha}")
print("─" * 70)


## 11. Contraste contra el endpoint del backend

El backend UBPD expone esta misma comparación en `GET /api/bi/comparison`. Contrastamos que los números coincidan.

In [ ]:
# Contraste contra el endpoint /api/bi/comparison
import requests

params = {
    "a_tipo":  SEL_A[0], "a_valor": SEL_A[1],
    "b_tipo":  SEL_B[0], "b_valor": SEL_B[1],
    "anio":    2026,
}
try:
    r = requests.get("http://localhost/api/bi/comparison", params=params, timeout=30)
    r.raise_for_status()
    api_resp = r.json()

    api_cmp = pd.DataFrame(api_resp["indicadores"])[
        ["codigo", "pct_a", "pct_b", "avance_a", "avance_b", "meta_a", "meta_b"]
    ].rename(columns={
        "codigo": "Código del Indicador",
        "pct_a":  "api_pct_a", "pct_b": "api_pct_b",
        "avance_a": "api_av_a", "avance_b": "api_av_b",
        "meta_a":   "api_mt_a", "meta_b":   "api_mt_b",
    })
    contraste = merged.merge(api_cmp, on="Código del Indicador", how="outer")
    contraste["dif_pct_a"] = (contraste["pct_a"] - contraste["api_pct_a"]).round(6)
    contraste["dif_pct_b"] = (contraste["pct_b"] - contraste["api_pct_b"]).round(6)
    print("BANNER API:  ", api_resp["banner"])
    print("HALLAZGO API:", api_resp["hallazgo"])
    print("RESUMEN API: ", api_resp["resumen"])
    print()
    print("Diferencias por indicador (deben ser 0):")
    contraste[["Código del Indicador", "pct_a", "api_pct_a", "dif_pct_a",
                                       "pct_b", "api_pct_b", "dif_pct_b"]]
except Exception as exc:
    print(f"⚠️  No se pudo contrastar contra la API: {exc}")


✅ Si `dif_a` y `dif_b` son todos **0**, significa que nuestro cálculo en pandas coincide exactamente con el que hace el backend al alimentar el BI.